# NoC Architecture Generation - Inference on Google Colab

This notebook generates NoC architectures from specifications using the trained Mistral-7B model.

**Setup Instructions:**
1. Upload this notebook to Google Colab
2. Enable GPU: Runtime → Change runtime type → T4 GPU
3. Upload trained model and test specifications
4. Run cells to generate architectures

## 1. Check GPU Availability

In [ ]:
# Check CUDA/GPU
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. Install Dependencies

In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate sentencepiece protobuf

## 3. Mount Google Drive (Optional but Recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust paths to your Drive structure
# !cp -r /content/drive/MyDrive/CaseStudy/outputs/mistral7b-noc-switch-qlora outputs/
# !cp -r /content/drive/MyDrive/CaseStudy/test_specs test_specs/

## 4. Create Directory Structure

In [ ]:
!mkdir -p src test_specs outputs/mistral7b-noc-switch-qlora outputs/generated

## 5. Create Validation Module

In [ ]:
%%writefile src/validate_architecture.py
#!/usr/bin/env python3
"""NoC Architecture Validation Module"""
import json
from typing import Dict, Tuple

class ArchitectureValidator:
    def __init__(self, spec: Dict, output: Dict):
        self.spec = spec
        self.output = output
        self.errors = []
        self.warnings = []
    
    def validate_all(self) -> Tuple[bool, Dict]:
        self.errors = []
        self.warnings = []
        
        self._validate_switch_placement()
        self._validate_path_elements()
        self._validate_route_connectivity()
        self._validate_no_cycles()
        
        is_valid = len(self.errors) == 0
        
        report = {
            "valid": is_valid,
            "errors": self.errors,
            "warnings": self.warnings,
            "checks": {
                "switch_placement": not self._has_error("switch_placement"),
                "path_elements": not self._has_error("path_elements"),
                "route_connectivity": not self._has_error("route_connectivity"),
                "no_cycles": not self._has_error("cycles")
            }
        }
        
        return is_valid, report
    
    def _has_error(self, error_type: str) -> bool:
        return any(error_type in e for e in self.errors)
    
    def _validate_switch_placement(self):
        floorplan = self.spec.get("floorplan_dim", [1000, 1000])
        max_x, max_y = floorplan[0], floorplan[1]
        blockages = self.spec.get("blockages", {})
        switches = self.output.get("switches", {})
        
        for switch_id, coords in switches.items():
            x, y = coords["x"], coords["y"]
            
            if not (0 <= x <= max_x and 0 <= y <= max_y):
                self.errors.append(
                    f"switch_placement: {switch_id} at ({x}, {y}) outside bounds ({max_x}, {max_y})"
                )
            
            for block_id, block in blockages.items():
                bx, by = block["x"], block["y"]
                bw, bh = block["width"], block["height"]
                
                if (bx <= x <= bx + bw) and (by <= y <= by + bh):
                    self.errors.append(
                        f"switch_placement: {switch_id} at ({x}, {y}) inside blockage {block_id}"
                    )
    
    def _validate_path_elements(self):
        inits = set(self.spec.get("inits", {}).keys())
        targets = set(self.spec.get("targets", {}).keys())
        switches = set(self.output.get("switches", {}).keys())
        
        all_valid_nodes = inits | targets | switches
        routing_paths = self.output.get("routing_paths", {})
        
        for route_id, path in routing_paths.items():
            if not isinstance(path, list):
                self.errors.append(f"path_elements: {route_id} path is not a list")
                continue
            
            for node in path:
                if node not in all_valid_nodes:
                    self.errors.append(
                        f"path_elements: {route_id} contains non-existent node '{node}'"
                    )
    
    def _validate_route_connectivity(self):
        connectivity = self.spec.get("connectivity", {})
        routing_paths = self.output.get("routing_paths", {})
        
        for route_id, (init, target) in connectivity.items():
            if route_id not in routing_paths:
                self.errors.append(
                    f"route_connectivity: Required route {route_id} ({init}->{target}) is missing"
                )
                continue
            
            path = routing_paths[route_id]
            
            if len(path) < 2:
                self.errors.append(
                    f"route_connectivity: {route_id} path too short"
                )
            elif path[0] != init:
                self.errors.append(
                    f"route_connectivity: {route_id} should start with {init}, got {path[0]}"
                )
            elif path[-1] != target:
                self.errors.append(
                    f"route_connectivity: {route_id} should end with {target}, got {path[-1]}"
                )
    
    def _validate_no_cycles(self):
        routing_paths = self.output.get("routing_paths", {})
        
        for route_id, path in routing_paths.items():
            if not isinstance(path, list) or len(path) < 2:
                continue
            
            seen_nodes = set()
            for node in path:
                if node in seen_nodes:
                    self.errors.append(
                        f"cycles: Route {route_id} contains loop - '{node}' appears multiple times"
                    )
                    break
                seen_nodes.add(node)

def validate_architecture(spec: Dict, output: Dict) -> Tuple[bool, Dict]:
    validator = ArchitectureValidator(spec, output)
    return validator.validate_all()

## 6. Create Inference Module

In [ ]:
%%writefile src/generate_architecture.py
#!/usr/bin/env python3
"""NoC Architecture Generation using Fine-tuned LLM"""
import json
import torch
import re
from typing import Dict, Optional, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from validate_architecture import validate_architecture

class NoCGenerator:
    def __init__(self, model_path: str, base_model: str = "mistralai/Mistral-7B-Instruct-v0.2"):
        print(f"Loading model from {model_path}...")
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        
        self.tokenizer = AutoTokenizer.from_pretrained(base_model)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        base = AutoModelForCausalLM.from_pretrained(
            base_model,
            quantization_config=bnb_config,
            device_map="auto"
        )
        
        self.model = PeftModel.from_pretrained(base, model_path)
        self.model.eval()
        
        print("✅ Model loaded successfully")
    
    def create_prompt(self, spec: Dict) -> str:
        spec_str = json.dumps(spec, indent=2)
        
        prompt = f"""You are an expert NoC (Network-on-Chip) physical designer. Given a chip floorplan specification including initiators, targets, connectivity requirements, and blockage zones, generate optimal switch placements and routing paths.

Output ONLY valid JSON with this structure:
{{
  "switches": {{"s_0": {{"x": <int>, "y": <int>}}, ...}},
  "routing_paths": {{"r_0": ["<init>", "<switches>", ..., "<target>"], ...}}
}}

Constraints:
- Switches must be within floorplan bounds and avoid blockage zones
- All routes in connectivity must have valid paths
- Network must be deadlock-free (no cycles)
- Minimize total wirelength

Specification:
{spec_str}

Generated Architecture:"""
        
        return prompt
    
    def generate(self, spec: Dict, max_new_tokens: int = 1024, temperature: float = 0.7, top_p: float = 0.9) -> Tuple[Optional[Dict], str]:
        prompt = self.create_prompt(spec)
        
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated_part = generated_text[len(prompt):].strip()
        output_dict = self._parse_output(generated_part)
        
        return output_dict, generated_part
    
    def _parse_output(self, text: str) -> Optional[Dict]:
        json_match = re.search(r'\{.*\}', text, re.DOTALL)
        
        if json_match:
            try:
                return json.loads(json_match.group(0))
            except json.JSONDecodeError as e:
                print(f"⚠️  JSON parsing error: {e}")
                return None
        
        print("⚠️  No valid JSON found in output")
        return None
    
    def generate_and_validate(self, spec: Dict, max_new_tokens: int = 1024, temperature: float = 0.7) -> Dict:
        output, raw_text = self.generate(spec, max_new_tokens, temperature)
        
        result = {
            "spec": spec,
            "output": output,
            "raw_generation": raw_text,
            "parsed_successfully": output is not None,
            "validation": None
        }
        
        if output is not None:
            is_valid, validation_report = validate_architecture(spec, output)
            result["validation"] = validation_report
            result["is_valid"] = is_valid
        else:
            result["is_valid"] = False
        
        return result

## 7. Upload Test Specification

Upload a test specification file or create one manually below.

In [ ]:
# Option 1: Upload files using Colab file upload
from google.colab import files

print("Upload your test specification (test_spec_XX.json):")
uploaded = files.upload()

# Move to test_specs directory
for filename in uploaded.keys():
    !mv {filename} test_specs/

In [ ]:
# Option 2: Create a simple test specification manually
test_spec = {
    "inits": {
        "i_0": {"x": 100, "y": 100},
        "i_1": {"x": 200, "y": 200}
    },
    "targets": {
        "t_0": {"x": 800, "y": 800},
        "t_1": {"x": 900, "y": 900}
    },
    "connectivity": {
        "r_0": ["i_0", "t_0"],
        "r_1": ["i_1", "t_1"],
        "r_2": ["i_0", "t_1"]
    },
    "floorplan_dim": [1000, 1000],
    "blockages": {}
}

# Save to file
import json
with open('test_specs/test_spec_manual.json', 'w') as f:
    json.dump(test_spec, f, indent=2)

print("✅ Test specification created")

## 8. Load Trained Model

**Important**: Make sure you've uploaded your trained model files to `outputs/mistral7b-noc-switch-qlora/`

In [ ]:
# Check model files exist
!ls -lh outputs/mistral7b-noc-switch-qlora/

In [ ]:
import sys
sys.path.insert(0, 'src')

from generate_architecture import NoCGenerator

# Load model
generator = NoCGenerator("outputs/mistral7b-noc-switch-qlora")

## 9. Generate Architecture - Single Specification

In [ ]:
import json

# Load specification
spec_file = 'test_specs/test_spec_manual.json'  # Change to your spec file

with open(spec_file) as f:
    spec = json.load(f)

print(f"Loaded specification:")
print(f"  Initiators: {len(spec['inits'])}")
print(f"  Targets: {len(spec['targets'])}")
print(f"  Routes: {len(spec['connectivity'])}")

In [ ]:
# Generate architecture
print("Generating architecture...\n")

result = generator.generate_and_validate(spec, temperature=0.7)

# Display results
print("="*60)
print("GENERATION RESULT")
print("="*60)
print(f"Parsed successfully: {result['parsed_successfully']}")
print(f"Valid architecture: {result['is_valid']}")

if result['parsed_successfully']:
    print(f"\nGenerated switches: {len(result['output'].get('switches', {}))}")
    print(f"Generated routes: {len(result['output'].get('routing_paths', {}))}")
    
    if result['validation']:
        print(f"\nValidation:")
        print(f"  Errors: {len(result['validation']['errors'])}")
        
        if result['validation']['errors']:
            print("\n  Error details:")
            for error in result['validation']['errors'][:5]:
                print(f"    - {error}")
        
        print(f"\n  Constraint checks:")
        for check, passed in result['validation']['checks'].items():
            status = "✅" if passed else "❌"
            print(f"    {status} {check}")
else:
    print("\n❌ Failed to parse JSON output")
    print(f"\nRaw output (first 300 chars):")
    print(result['raw_generation'][:300])

## 10. Display Generated Architecture

In [ ]:
if result['parsed_successfully'] and result['output']:
    print("Generated Architecture:\n")
    print(json.dumps(result['output'], indent=2))

## 11. Visualize Architecture

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

if result['parsed_successfully'] and result['output']:
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Get floorplan dimensions
    max_x, max_y = spec['floorplan_dim']
    
    # Draw floorplan boundary
    ax.add_patch(patches.Rectangle((0, 0), max_x, max_y, 
                                    fill=False, edgecolor='black', linewidth=2))
    
    # Draw blockages
    for block_id, block in spec.get('blockages', {}).items():
        ax.add_patch(patches.Rectangle(
            (block['x'], block['y']), block['width'], block['height'],
            fill=True, facecolor='gray', alpha=0.5, edgecolor='black'
        ))
    
    # Draw initiators
    for init_id, coords in spec['inits'].items():
        ax.plot(coords['x'], coords['y'], 'go', markersize=12, label='Initiator' if init_id == list(spec['inits'].keys())[0] else '')
        ax.text(coords['x'], coords['y']+30, init_id, ha='center', fontsize=8)
    
    # Draw targets
    for target_id, coords in spec['targets'].items():
        ax.plot(coords['x'], coords['y'], 'ro', markersize=12, label='Target' if target_id == list(spec['targets'].keys())[0] else '')
        ax.text(coords['x'], coords['y']+30, target_id, ha='center', fontsize=8)
    
    # Draw switches
    for switch_id, coords in result['output'].get('switches', {}).items():
        ax.plot(coords['x'], coords['y'], 'bs', markersize=10, label='Switch' if switch_id == list(result['output']['switches'].keys())[0] else '')
        ax.text(coords['x'], coords['y']+30, switch_id, ha='center', fontsize=8)
    
    ax.set_xlim(-50, max_x + 50)
    ax.set_ylim(-50, max_y + 50)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('X coordinate')
    ax.set_ylabel('Y coordinate')
    ax.set_title('Generated NoC Architecture')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print routing paths
    print("\nRouting Paths:")
    for route_id, path in result['output'].get('routing_paths', {}).items():
        print(f"  {route_id}: {' → '.join(path)}")
else:
    print("Cannot visualize - no valid output")

## 12. Batch Generation (Multiple Specifications)

In [ ]:
# List all test specs
import os
from pathlib import Path

spec_files = sorted(Path('test_specs').glob('test_spec_*.json'))
print(f"Found {len(spec_files)} test specifications")

for spec_file in spec_files:
    print(f"  - {spec_file.name}")

In [ ]:
# Generate for all specs
batch_results = []
stats = {"total": 0, "parsed": 0, "valid": 0, "invalid": 0}

for i, spec_file in enumerate(spec_files[:5]):  # Limit to first 5
    print(f"\nGenerating {i+1}/{min(5, len(spec_files))}: {spec_file.name}")
    
    with open(spec_file) as f:
        spec = json.load(f)
    
    result = generator.generate_and_validate(spec, temperature=0.7)
    batch_results.append(result)
    
    # Update stats
    stats["total"] += 1
    if result["parsed_successfully"]:
        stats["parsed"] += 1
        if result["is_valid"]:
            stats["valid"] += 1
            print("  ✅ Valid")
        else:
            stats["invalid"] += 1
            print(f"  ❌ Invalid ({len(result['validation']['errors'])} errors)")
    else:
        print("  ⚠️  Parse failed")
    
    # Save result
    output_file = f"outputs/generated/result_{i:03d}.json"
    with open(output_file, 'w') as f:
        json.dump(result, f, indent=2)

# Print summary
print("\n" + "="*60)
print("BATCH GENERATION SUMMARY")
print("="*60)
print(f"Total: {stats['total']}")
print(f"Parsed: {stats['parsed']} ({100*stats['parsed']/stats['total']:.1f}%)")
print(f"Valid: {stats['valid']} ({100*stats['valid']/stats['total']:.1f}%)")
print(f"Invalid: {stats['invalid']}")

## 13. Analyze Results

In [ ]:
# Constraint pass rates
if batch_results:
    check_stats = {
        'switch_placement': 0,
        'path_elements': 0,
        'route_connectivity': 0,
        'no_cycles': 0
    }
    
    valid_results = [r for r in batch_results if r.get('validation')]
    
    for result in valid_results:
        for check_name, passed in result['validation']['checks'].items():
            if passed:
                check_stats[check_name] += 1
    
    print("Constraint Pass Rates:")
    for check_name, count in check_stats.items():
        if valid_results:
            rate = 100 * count / len(valid_results)
            print(f"  {check_name}: {count}/{len(valid_results)} ({rate:.1f}%)")

In [ ]:
# Error distribution
if batch_results:
    error_types = {}
    
    for result in batch_results:
        if result.get('validation') and result['validation']['errors']:
            for error in result['validation']['errors']:
                error_type = error.split(':')[0]
                error_types[error_type] = error_types.get(error_type, 0) + 1
    
    if error_types:
        print("\nError Distribution:")
        for error_type, count in sorted(error_types.items(), key=lambda x: x[1], reverse=True):
            print(f"  {error_type}: {count}")
    else:
        print("\n✅ No errors found!")

## 14. Visualize Statistics

In [ ]:
if batch_results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Validity distribution
    ax1.pie(
        [stats['valid'], stats['invalid'], stats['total'] - stats['parsed']],
        labels=['Valid', 'Invalid', 'Parse Failed'],
        autopct='%1.1f%%',
        colors=['#2ecc71', '#e74c3c', '#95a5a6'],
        startangle=90
    )
    ax1.set_title('Generation Results Distribution')
    
    # Constraint pass rates
    if valid_results:
        checks = list(check_stats.keys())
        pass_rates = [100 * check_stats[c] / len(valid_results) for c in checks]
        
        ax2.barh(checks, pass_rates, color='#3498db')
        ax2.set_xlabel('Pass Rate (%)')
        ax2.set_title('Constraint Validation Pass Rates')
        ax2.set_xlim(0, 100)
        ax2.grid(axis='x', alpha=0.3)
        
        for i, v in enumerate(pass_rates):
            ax2.text(v + 1, i, f'{v:.1f}%', va='center')
    
    plt.tight_layout()
    plt.show()

## 15. Download Results

In [ ]:
# Create zip of all results
!zip -r generation_results.zip outputs/generated/

# Download
from google.colab import files
files.download('generation_results.zip')

## Summary

This notebook demonstrates:
- ✅ Loading trained NoC generation model
- ✅ Generating architectures from specifications
- ✅ Automatic validation with constraint checking
- ✅ Visualization of generated architectures
- ✅ Batch processing multiple specifications
- ✅ Statistical analysis and metrics

**Key Metrics:**
- **Parsing Rate**: Percentage of valid JSON outputs
- **Validity Rate**: Percentage passing all constraints
- **Constraint Pass Rates**: Per-constraint success rates

Use these results to:
1. Assess model quality
2. Identify improvement areas
3. Compare different checkpoints
4. Generate case study examples